# vla-hands · Intro Notebook

**Goal:** Turn a frozen vision-language model into a robot controller by grafting a tiny action head onto its hidden states.

This intro notebook does exactly one thing end-to-end:

1. Load **SmolVLM-256M** (the smallest capable VLM, ~1 GB)
2. Attach a **JoystickAppendage** — a small MLP that reads the VLM's last hidden state and outputs `(dx, dy) ∈ [-1, 1]²`
3. Train it on **TargetNav** — a toy task where an agent must navigate to a coloured dot
4. Save the appendage weights (~100 KB) and reload them

**Runtime:** ~5 min on Colab Free T4 (CPU works too, just slower)  
**No GPU required** for the quick demo — though it helps.

## 1 · Install

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza
print('✅ vla-hands installed')

In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import display

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2 · The Environment

In [ ]:
from vla_hands import TargetNavEnvironment

env = TargetNavEnvironment(width=224, height=224, max_steps=80)

# Each reset() samples a new prompt from a synonym vocabulary
for i in range(3):
    obs = env.reset(seed=i)
    print(f'seed={i}: {env.prompt}')

# Show a reset observation
obs = env.reset(seed=42)
plt.figure(figsize=(4, 4))
plt.imshow(obs)
plt.title('TargetNav — navigate the agent (blue) to the target (red)')
plt.axis('off')
plt.show()

In [ ]:
# Watch the expert policy solve a few episodes
import numpy as np

env = TargetNavEnvironment(width=224, height=224, max_steps=80, speed=20.0)
obs = env.reset(seed=7)
frames = [obs]
for _ in range(11):
    result = env.step(env.expert_action())
    frames.append(result.observation)
    if result.done:
        break

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, frame in zip(axes.flat, frames):
    ax.imshow(frame)
    ax.axis('off')
for ax in list(axes.flat)[len(frames):]:
    ax.set_visible(False)
plt.suptitle('Expert policy rollout (upper bound for training)')
plt.tight_layout()
plt.show()
print(f'Episode ended: success={result.info.get("success")}, reward={result.reward:.1f}')

## 3 · Load the VLM

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'

print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)

# SmolVLM stores language hidden size in text_config
text_cfg = getattr(vlm.config, 'text_config', None)
hidden_dim = (
    text_cfg.hidden_size
    if text_cfg is not None and hasattr(text_cfg, 'hidden_size')
    else vlm.config.hidden_size
)

n_params = sum(p.numel() for p in vlm.parameters())
print(f'Parameters:  {n_params/1e6:.0f}M')
print(f'Hidden dim:  {hidden_dim}')

## 4 · Build the Graft

In [ ]:
from vla_hands import VLAGraft, GraftConfig, JoystickAppendage

# The appendage is a tiny MLP (~50K params):
#   hidden_state → LayerNorm → Linear → GELU → Linear → Tanh → (dx, dy)
appendage = JoystickAppendage(hidden_dim=hidden_dim)

# VLAGraft pairs the VLM backbone with the appendage.
# The backbone is frozen; only appendage weights are trained by default.
graft = VLAGraft(
    vlm=vlm,
    appendage=appendage,
    config=GraftConfig(feature_extraction='last'),  # use last-token hidden state
)

appendage_params = appendage.num_parameters()
print(graft)
print(f'\nAppendage params: {appendage_params:,}  ({appendage_params/n_params*100:.3f}% of VLM)')

## 5 · Train

Two phases:
- **BC (Behavioral Cloning):** Supervised imitation of the expert policy. Fast, stable.
- **RL (REINFORCE):** Fine-tune with environment reward. Corrects distributional shift.

For this intro we keep it short — the goal is to verify the pipeline works.

In [ ]:
from vla_hands import TrainingCurriculum, CurriculumConfig, QUICK_CURRICULUM

config = CurriculumConfig(
    bc_steps=200,          # imitation learning warm-start
    rl_steps=50,           # brief RL fine-tuning
    device=device,
    save_dir='model_checkpoints/intro_joystick',
    freezing_stages=QUICK_CURRICULUM,  # keep VLM fully frozen (fastest)
    eval_every=50,
    log_every=20,
    eval_episodes=5,
)

curriculum = TrainingCurriculum(
    graft=graft,
    processor=processor,
    environment=env,
    config=config,
)

metrics = curriculum.run()
print('\nTraining complete!')

## 6 · Evaluate

In [ ]:
bc_metrics = metrics.get('bc', metrics)
steps = [m['step'] for m in bc_metrics if 'bc/loss' in m]
losses = [m['bc/loss'] for m in bc_metrics if 'bc/loss' in m]

plt.figure(figsize=(8, 3))
plt.plot(steps, losses, linewidth=2)
plt.xlabel('Step')
plt.ylabel('BC Loss')
plt.title('Behavioral Cloning Training Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final BC loss: {losses[-1]:.4f}' if losses else 'No loss data.')

In [ ]:
from vla_hands import BenchmarkSuite

eval_env = TargetNavEnvironment(width=224, height=224, max_steps=80)
suite = BenchmarkSuite(graft, processor, [eval_env], device=device)
result = suite.run_benchmark(eval_env, n_episodes=20)

print(f'Success rate: {result.success_rate:.0%}')
print(f'Mean reward:  {result.mean_reward:+.2f}')
print(result)

## 7 · Save & Reload

In [ ]:
import os, json

# Save — only the appendage MLP weights are stored (~100 KB)
graft.save('model_checkpoints/intro_joystick')
files = os.listdir('model_checkpoints/intro_joystick')
sizes = {f: os.path.getsize(f'model_checkpoints/intro_joystick/{f}') for f in files}
print(f'Saved files: {files}')
print(f'Total size:  {sum(sizes.values())/1024:.1f} KB')

# Reload into a fresh graft (VLM is loaded from HuggingFace Hub separately)
fresh_graft = VLAGraft(
    vlm=vlm,
    appendage=JoystickAppendage(hidden_dim),
    config=GraftConfig(feature_extraction='last'),
)
fresh_graft.load_appendage('model_checkpoints/intro_joystick')
print('\nReload successful!')

# One-shot loader (from_pretrained)
fresh_graft2 = VLAGraft.from_pretrained(
    vlm_id=MODEL_ID,
    appendage=JoystickAppendage(hidden_dim),
    checkpoint_path='model_checkpoints/intro_joystick',
    device=device,
)
print('from_pretrained() reload also works!')

## ✅ Done!

You've trained your first VLA graft. What happened under the hood:

- The VLM processed `(image, prompt)` pairs and output hidden states
- The JoystickAppendage read the **last-token hidden state** and learned to map it to `(dx, dy)`
- BC trained it to imitate the expert; RL fine-tuned it with actual environment rewards
- Only ~100 KB of appendage weights were saved — the VLM backbone is unchanged

**Next steps:**
- `02_medium.ipynb` — all 5 appendage types, training curves, GIF export, benchmark suite
- `03_advanced.ipynb` — LoRA, CompositeGraft (multi-appendage), auto curriculum, new environments

## 8 · Export a GIF

Visualise what your trained graft actually does — no ffmpeg required.

In [ ]:
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif
from IPython.display import Image as IPImage
import os

os.makedirs('gifs', exist_ok=True)

# Expert baseline GIF — what a perfect agent looks like
record_expert_gif(
    env=TargetNavEnvironment(width=200, height=200),
    path='gifs/expert_targetnav.gif',
    n_steps=20, seed=7, fps=8,
)
print('Expert GIF saved.')
IPImage(filename='gifs/expert_targetnav.gif')

In [ ]:
# Trained graft GIF — shows your model's behaviour after training
save_rollout_gif(
    graft=graft,
    processor=processor,
    env=TargetNavEnvironment(width=200, height=200),
    path='gifs/trained_joystick.gif',
    n_steps=20, seed=42, device=device, fps=6,
)
print('Trained graft GIF saved.')
IPImage(filename='gifs/trained_joystick.gif')